# Simulate an order update

Changes one in-transit order in Lakebase, so the next extract and pipeline run can be checked end to end.

**What it does**
- Picks one order with status `shipped`
- Marks it as delivered two days after its promised date (an SLA breach)
- The trigger bumps `updated_at`, so the next extract picks up exactly that row

**Good to know**
- This changes the source data. Use it for demos and tests only.

In [0]:
%pip install -U databricks-sdk "psycopg[binary]"

In [0]:
dbutils.library.restartPython()

In [0]:
# Lakebase connection settings (same as 01_load_lakebase).
ENDPOINT  = "projects/olist-capstone/branches/production/endpoints/primary"
HOST      = "ep-noisy-grass-d1gvpml8.database.us-west-2.cloud.databricks.com"
DB        = "databricks_postgres"
PG_SCHEMA = "olist"
USER      = spark.sql("select current_user()").first()[0]

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

def pg_token() -> str:
    """Short-lived OAuth token (1h) used as the Postgres password."""
    return w.postgres.generate_database_credential(endpoint=ENDPOINT).token

In [0]:
# Deliver one shipped order two days after its promised date.
import psycopg

with psycopg.connect(host=HOST, port=5432, dbname=DB, user=USER,
                     password=pg_token(), sslmode="require") as conn, conn.cursor() as cur:
    cur.execute(f"""
        update {PG_SCHEMA}.orders
           set order_status = 'delivered',
               order_delivered_customer_date = order_estimated_delivery_date + interval '2 days'
         where order_id = (select order_id from {PG_SCHEMA}.orders
                            where order_status = 'shipped' limit 1)
     returning order_id""")
    print("Updated order:", cur.fetchone()[0])
    conn.commit()